# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook illustrates how to load, explore, and analyze a multi-record-set Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is published at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed!pip install mlcroissant

## 1. Data Loading
Load and inspect the metadata and structure of the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}:\n{metadata.description}\n")
print(f"Citation: {metadata.citeAs}")


## 2. Data Overview
Let's list all available record sets in the dataset and their fields by `@id`.

_Note: All references use the `@id` identifiers as per Croissant best practices._

In [ ]:
# List and display all record sets, their @id, and main field @ids
record_sets = [rs for rs in dataset.record_sets]
if len(record_sets) == 0:
    print('No record sets defined in this dataset. Please check the Croissant schema for available data.')
else:
    for rs in record_sets:
        print(f"---\nRecord set name: {rs.name}")
        print(f"@id: {rs.id}")
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - {fld.name} (@id: {fld.id}, type: {fld.data_type})")
        print()


## 3. Data Extraction
Load the records from a specific record set (by `@id`) into a DataFrame for further exploration.

> Update the `record_set_id` variable to match the `@id` of the record set you wish to explore (as printed in the previous overview).

In [ ]:
# If available, extract all record sets into pandas DataFrames (using @id for variable keys)
dataframes = {}

# Collect @ids for all record sets
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs in dataset.record_sets:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs.id))
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records from record set: {rs.name} (@id: {rs.id})")
    except Exception as ex:
        print(f"Record set {rs.id} could not be loaded: {ex}")

# Preview columns for first available record set
if dataframes:
    selected_rs_id = list(dataframes.keys())[0]
    print(f"\nAvailable fields in DataFrame for record set '@id': {selected_rs_id}")
    print(dataframes[selected_rs_id].columns.tolist())
    dataframes[selected_rs_id].head()
else:
    print('No dataframes available. Check that the Croissant schema includes record sets and data files.')

## 4. Exploratory Data Analysis (EDA)
Perform simple filtering and transformation on one numeric field and group the data by a specified key field. All column and field references use the corresponding `@id`.

> Set `record_set_id`, `numeric_field_id`, and `group_field_id` to match the `@id`s from the loaded data. Below, they are set dynamically if detected, but you can override them based on your exploration.

In [ ]:
# Pick a record set with data for demonstration
if not dataframes:
    print('No dataframes to analyze.')
else:
    # Select primary record set (first in list)
    record_set_id = selected_rs_id
    df = dataframes[record_set_id]

    # Try to auto-detect a numeric field (@id that looks like regression output or numeric measurement)
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in ('i', 'u', 'f')]
    if not numeric_candidates:
        print('No numeric fields detected in the DataFrame.')
    else:
        numeric_field_id = numeric_candidates[0]  # Use the first numeric @id
        print(f"Numeric field used for EDA: {numeric_field_id}")

        # Filter records with value above the mean
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized column: {norm_col}")

        # Try to group by a likely field (categorical @id)
        group_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped.head())
        else:
            print('No categorical fields to group by detected.')

        # Display sample transformed data
        print(filtered_df[[numeric_field_id, norm_col]].head())

## 5. Visualization
Visualize the distribution of a numeric variable, and a grouped bar for the mean value by a chosen categorical field. All axes and references use Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_candidates:
    print('No data for plotting.')
else:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id exists, plot mean numeric value by group
    if group_candidates:
        group_field_id = group_candidates[0]
        plt.figure(figsize=(8, 4))
        # Compute means
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=means.index.astype(str), y=means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded dataset metadata and structure from a Croissant schema.
- Listed available record sets and explored their fields using `@id` references.
- Loaded rows from record sets into pandas DataFrames.
- Performed basic EDA: filtering, normalization, and grouping by categorical field.
- Visualized numeric distributions and group means.

The FAIR^2 Croissant model enables rich, structured access to machine-actionable datasets and supports object-oriented exploration using stable identifiers for all records and fields. To go further, readers can apply machine learning, deep-dive on variable relationships, or map `@id`s to domain-specific knowledge graphs.
